[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/03_clustering.ipynb)

# Capítulo 3. Clustering de textos con K-Means

Este notebook forma parte del apunte del curso **Analítica Textual** y está preparado para ejecutarse en Google Colab o en Jupyter local.


In [ ]:
# Setup opcional para Colab
# En general, este notebook corre sin instalaciones adicionales.
# Si tu entorno no tiene las librerías base, descomenta la línea siguiente.
# !pip -q install scikit-learn pandas


## Objetivos

En esta semana nos movemos a aprendizaje no supervisado. Buscamos estructura sin etiquetas previas.

Al finalizar deberías poder:

- explicar qué problema resuelve el clustering;
- aplicar K-Means sobre textos vectorizados;
- inspeccionar centroides para interpretar grupos;
- reconocer límites y riesgos de una segmentación automática.

## 3.1 ¿Para qué agrupar textos?

El clustering sirve para:

- descubrir temas dominantes en un conjunto documental;
- organizar archivos o tickets;
- detectar subpoblaciones antes de construir modelos supervisados;
- resumir grandes colecciones para exploración inicial.

## 3.2 Pipeline general

Un flujo típico es:

1. limpiar texto;
2. vectorizar con TF-IDF;
3. elegir número de clusters `k`;
4. entrenar K-Means;
5. interpretar resultados.

## 3.3 Corpus de ejemplo


In [ ]:
documentos = [
    "el equipo ganó el partido de fútbol",
    "el entrenador prepara la temporada deportiva",
    "la selección necesita mejores defensas",
    "el congreso debate una nueva reforma tributaria",
    "el ministerio anunció cambios regulatorios",
    "el parlamento votará el presupuesto nacional",
    "la empresa desarrolla software de visión computacional",
    "el nuevo modelo usa datos para automatizar procesos",
    "la plataforma cloud reduce tiempos de despliegue"
]


## 3.4 Vectorización y entrenamiento


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd

vectorizador = TfidfVectorizer(stop_words=None)
X = vectorizador.fit_transform(documentos)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

pd.DataFrame({"documento": documentos, "cluster": clusters})


## 3.5 Interpretación de clusters

K-Means entrega centroides. Si miramos las palabras con mayor peso en cada centroide, obtenemos una aproximación al tema del grupo.


In [ ]:
terminos = vectorizador.get_feature_names_out()

for i, centroide in enumerate(kmeans.cluster_centers_):
    top = centroide.argsort()[-5:][::-1]
    palabras = [terminos[j] for j in top]
    print(f"Cluster {i}: {', '.join(palabras)}")


Esta inspección es fundamental porque los clusters no vienen etiquetados.

## 3.6 ¿Cómo elegir `k`?

No existe una respuesta universal. Algunas estrategias:

- conocimiento del dominio;
- prueba de varios valores y comparación cualitativa;
- métrica de silueta;
- costo de interpretación para el equipo de negocio.

Ejemplo con silueta:


In [ ]:
from sklearn.metrics import silhouette_score

for k in range(2, 6):
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = modelo.fit_predict(X)
    score = silhouette_score(X, etiquetas)
    print(k, round(score, 3))


## 3.7 Limitaciones de K-Means en texto

Aunque es muy usado, K-Means tiene supuestos fuertes:

- favorece clusters aproximadamente esféricos;
- depende del valor de `k`;
- es sensible a inicialización y representación vectorial;
- puede producir grupos difíciles de explicar.

En texto, muchas veces el mayor desafío no es ejecutar el algoritmo sino dar sentido a los grupos.

## 3.8 Uso aplicado

Imagina una mesa de ayuda con miles de tickets sin clasificar. Un clustering preliminar puede revelar grupos como:

- problemas de facturación;
- incidencias técnicas;
- consultas de acceso;
- solicitudes comerciales.

Eso permite diseñar etiquetas, flujos de atención y futuros modelos supervisados.

## Ejercicios

1. Cambia `n_clusters` a 2 y luego a 4. ¿Qué cambia en la interpretabilidad?
2. Reemplaza el corpus por opiniones de productos o noticias.
3. Usa bigramas en el vectorizador y compara los términos representativos.

## Idea clave

El clustering no reemplaza al juicio humano. Su valor está en acelerar exploración, descubrir estructura y ayudar a formular mejores preguntas sobre los datos.
